# 第 08 章 · 权限与审批：把意图变成执行许可

“用户同意了”不能只有一个布尔值。需要知道同意的是哪个动作、哪份参数、哪个环境、哪种范围和哪个政策版本。
等待期间参数或政策变化，旧同意不应继续生效；执行器不具备相应限制能力时，同意也不能制造这种能力。

本章区分 **评审路线 → 决策来源 → Permit → 执行前匹配 → 有限重试**。
Scope 中 `note`、`log`、`secret` 是精确资源 ID，不能当作已解析的 OS 路径。
我们在显式门控之后实际写一个本地文件，验证错误决策没有进入 handler；路径强制执行仍属于下一章的真实执行器。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
approval = load_example('08', 'policy')
from dataclasses import replace
import json
run_dir = fresh_run_dir('ch08')
target = run_dir / 'note.txt'
entries = []
def execute_if_matched(permit, action, current_policy, supported, *, cancelled=False):
    verdict = approval.check(permit, action, current_policy, supported, cancelled=cancelled)
    if verdict == 'matched':
        entries.append(action.call)
        target.write_text(json.loads(action.arguments_json)['text'], encoding='utf-8')
    return verdict

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. Permit 绑定动作，而不是工具名


    同一个写文件工具可以写完全不同的正文。动作摘要对规范化 JSON、环境和操作计算，Permit 还绑定 session、call、
    政策版本和作用范围。调整正文之后需要重新产生与该正文相符的许可。


In [3]:
text_to_write = '已复核筛选条件。'
policy = approval.Policy('policy-1', managed_deny=frozenset({'secret'}))
scope = approval.Scope(write=frozenset({'note'}), deny=frozenset({'secret'}))
supported = approval.Scope(write=frozenset({'note', 'log'}), deny=frozenset({'secret'}))
action = approval.Action('session-1', 'call-1', 'local-example', 'write_note',
                         json.dumps({'text': text_to_write}), scope)
permit = approval.issue(action, policy, approval.Requirement.REVIEW, approval.Source.USER, 'fixture-user')
assert execute_if_matched(permit, action, policy, supported) == 'matched'
assert target.read_text() == text_to_write and entries == ['call-1']
{'source': permit.source.value, 'policy_version': permit.policy_version, 'action_key': permit.action_key, 'written_bytes': target.stat().st_size}


{'source': 'user',
 'policy_version': 'policy-1',
 'action_key': '020210ff0ee8e62a0bd107a8a2d03e5bee050bcea1a959da78d44f1331702a4e',
 'written_bytes': 24}

## 2. 参数、政策、范围任何一项变化都要重核


    许可匹配和执行能力是两层检查。下面依次改变正文、政策版本、请求范围和后端能力。
    匹配失败返回 mismatch，后端不能实施原许可则返回 unsupported；两者都不得进入 handler。
    许可范围包含 deny，后端声称支持写入却丢了 deny，也不能执行。


In [4]:
count, before = len(entries), target.read_bytes()
changed_text = replace(action, arguments_json=json.dumps({'text': '未经同意的内容'}))
widened = replace(action, scope=replace(scope, write=frozenset({'note', 'log'})))
changed_policy = replace(policy, version='policy-2')
weak_backend = approval.Scope(write=frozenset({'note'}))  # 无法保留 required deny
verdicts = [
    execute_if_matched(permit, changed_text, policy, supported),
    execute_if_matched(permit, action, changed_policy, supported),
    execute_if_matched(permit, widened, policy, supported),
    execute_if_matched(permit, action, policy, weak_backend),
]
assert verdicts == ['mismatch', 'mismatch', 'mismatch', 'unsupported']
assert len(entries) == count and target.read_bytes() == before
verdicts


['mismatch', 'mismatch', 'mismatch', 'unsupported']

## 3. 强制人工要求不能被自动 allow 替代


    决策来源影响它能覆盖什么。Hook 的 allow、Guardian 的判断和用户的一次批准不必具有相同权限。
    `human_once` 要求当次人工决策；如果政策同时强制自动评审，不能通过猜测来化解矛盾，应明确拒绝这组配置。
    deny 的优先级还必须独立验证。


In [5]:
assert approval.route(approval.Requirement.REVIEW, policy, ['allow', 'deny']) == 'deny'
assert approval.route(approval.Requirement.HUMAN, policy, ['allow']) == 'user'
strict = replace(policy, strict_auto=True)
assert approval.route(approval.Requirement.HUMAN, strict) == 'incompatible_review_requirements'
automated = expect_error(ValueError, lambda: approval.issue(action, policy, approval.Requirement.HUMAN, approval.Source.HOOK, 'hook'), 'fresh human')
reusable = expect_error(ValueError, lambda: approval.issue(action, policy, approval.Requirement.HUMAN, approval.Source.USER, 'user', session_wide=True), 'fresh human')
human = approval.issue(action, policy, approval.Requirement.HUMAN, approval.Source.USER, 'user')
assert human.call == action.call
forbidden_action = replace(action, scope=approval.Scope(write=frozenset({'secret'})))
denied = expect_error(ValueError, lambda: approval.issue(forbidden_action, policy, approval.Requirement.REVIEW, approval.Source.USER, 'user'), 'managed deny')
{'automated_checkpoint': automated, 'reusable_checkpoint': reusable, 'managed_rule': denied}


{'automated_checkpoint': 'fresh human decision required',
 'reusable_checkpoint': 'fresh human decision required',
 'managed_rule': 'managed deny'}

## 4. 等待取消之后，迟到批准只能成为历史


    pending owner 控制等待状态。取消后到达的答案不能签发 Permit；若批准先到、取消后到，执行前检查仍要拦住动作。
    这两个顺序都需要验证，单测其中一个不足以说明取消边界。


In [6]:
pending = approval.Pending('review-1', action, policy.version)
cancelled = pending.cancel()
state, late_permit = cancelled.resolve('review-1', action, policy, approval.Source.USER, 'user')
assert state.state == 'cancelled' and late_permit is None
resolved, early_permit = pending.resolve('review-1', action, policy, approval.Source.USER, 'user')
assert resolved.state == 'resolved' and early_permit is not None
count = len(entries)
assert execute_if_matched(early_permit, action, policy, supported, cancelled=True) == 'cancelled'
assert len(entries) == count
duplicate_state, duplicate_permit = resolved.resolve('review-1', action, policy, approval.Source.USER, 'user')
assert duplicate_permit is None
{'cancel_then_answer': state.state, 'answer_then_cancel': 'blocked_before_handler', 'duplicate_answer_permit': duplicate_permit}


{'cancel_then_answer': 'cancelled',
 'answer_then_cancel': 'blocked_before_handler',
 'duplicate_answer_permit': None}

## 5. 重试不能把一次普通失败升级成扩权


    普通非零退出可能是程序 bug，扩大权限不会让它合理。只有结构化 sandbox_denied 才进入本原型的扩权规划。
    User/Hook 的显式 retry_scope 可以授权有限范围；Guardian 的首次批准不能自动当成更宽动作的许可。
    这里仅计算路线，不发起无沙箱执行；网络审批仍交给网络 owner。


In [7]:
retry_ceiling = replace(scope, write=frozenset({'note', 'log'}))
user_permit = approval.issue(action, policy, approval.Requirement.REVIEW, approval.Source.USER,
                             'user', retry_scope=retry_ceiling)
wider_action = replace(action, scope=retry_ceiling)
ordinary = approval.RetryContext('exit_nonzero', 'unless_trusted')
sandbox_failure = approval.RetryContext('sandbox_denied', 'unless_trusted')
stopped, _ = approval.plan_retry(user_permit, wider_action, policy, supported, ordinary)
reused, derived = approval.plan_retry(user_permit, wider_action, policy, supported, sandbox_failure)
guardian = approval.issue(action, policy, approval.Requirement.REVIEW, approval.Source.GUARDIAN, 'guardian')
reviewed, _ = approval.plan_retry(guardian, wider_action, policy, supported, sandbox_failure)
network_route, _ = approval.plan_retry(user_permit, wider_action, policy, supported, replace(sandbox_failure, network_review=True))
assert stopped == 'stop:not_retryable'
assert reused == 'reuse' and derived.scope == retry_ceiling
assert reviewed == 'review:actual_retry'
assert network_route == 'review:network_owner'
{'ordinary_failure': stopped, 'explicit_user_retry': reused, 'guardian_retry': reviewed, 'network_retry': network_route}


{'ordinary_failure': 'stop:not_retryable',
 'explicit_user_retry': 'reuse',
 'guardian_retry': 'review:actual_retry',
 'network_retry': 'review:network_owner'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [check_approval_requirement / request_reviewer_approval / request_user_approval](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/approvals.rs#L525-L547)：managed human checkpoint 不能由 Hook、Guardian 或会话缓存替代；若强制自动审查与必须人工冲突则拒绝。

- [ApprovalAction::cache_keys](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/approvals.rs#L284-L320)：命令批准缓存键绑定 environment、可执行项、规范化命令、cwd、tty、sandbox_permissions、additional_permissions；用户路径再叠环境 exec policy fingerprint。apply_patch 的键粒度不同，为环境与文件。

- [sandbox denied retry guards](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/orchestrator.rs#L401-L479)：只有 SandboxErr::Denied 可进入此扩权分支；Never/OnRequest 默认不做无沙箱重试，OnRequest 的特定 managed-network 审批是单独例外。


## 现在应能解释什么


    能说明批准绑定的具体内容，以及审批决定、动作匹配和执行器能力为何缺一不可。
    接回 Dispatcher 时在最终动作产生后评审，在启动前再次检查取消、版本、scope；不要重新建立另一套调用身份。

    本章复用局部政策原型，实际文件写入只验证门控前后行为；没有真实审批 UI、政策文件解析或 OS 级权限强制。
